In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from utils.tokenizer import Tokenizer

df = pd.read_csv("../flickr8k/captions.txt")

images = df["image"].unique()

train_images, test_images = train_test_split(
    images,
    test_size=0.05,
    random_state=42,
    shuffle=True
)

train_df = df[df["image"].isin(train_images)]
test_df = df[df["image"].isin(test_images)]

print(f"Train images: {len(train_images)}")
print(f"Test images: {len(test_images)}")

print(f"Train captions: {len(train_df)}")
print(f"Test captions: {len(test_df)}")

Train images: 7686
Test images: 405
Train captions: 38430
Test captions: 2025


In [2]:
import json
import pickle

from utils.text_cleaner import clean_all
from utils.vocabulary import Vocabulary
from utils.max_len import compute_max_len

train_captions: list[str] = train_df["caption"].tolist()
test_captions: list[str] = test_df["caption"].tolist()

clean_train_captions = clean_all(train_captions)
test_captions = clean_all(test_captions)

tokenizer = Tokenizer()

vocabulary = Vocabulary(tokenizer)
vocabulary.build(train_captions)

with open("../config/vocab.pkl", "wb") as f:
    pickle.dump(vocabulary, f)

tok_train_captions: list[list[str]] = tokenizer.tokenize_all(clean_train_captions)
max_len = compute_max_len(tok_train_captions)

config = { "max_len_without_start_and_end_tokens": max_len }

with open("../config/vectorizer_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

print(f"Vocab size: {vocabulary.size}")
print(f"Max caption len: {max_len}")

train_df["caption"] = clean_train_captions
test_df["caption"] = test_captions

train_df.to_csv("../flickr8k/captions_train.csv", index=False)
test_df.to_csv("../flickr8k/captions_test.csv", index=False)

Vocab size: 3016
Max caption len: 18
